In [1]:
import json
import csv
import statistics

with open('aqi_hourly_raw.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

with open('aqi_tracker.csv', 'r', newline='', encoding='utf-8') as f:
    tracker_rows = list(csv.DictReader(f))

volatility = {}
above_60_count = 0

for city, readings in raw_data.items():
    valid = [val for val in readings if val is not None]
    volatility[city] = round(statistics.pstdev(valid), 2)
    for val in valid:
        if val > 60:
            above_60_count += 1

most_volatile_city = max(volatility, key=volatility.get)

city_avgs = {r['city']: float(r['avg_pm25']) for r in tracker_rows}
mean_of_avgs = statistics.mean(city_avgs.values())
below_mean_cities = [c for c, avg in city_avgs.items() if avg < mean_of_avgs]

insights = {
    'most_volatile_city': most_volatile_city,
    'volatility_pstdev': volatility,
    'mean_of_city_averages': round(mean_of_avgs, 2),
    'cities_below_mean_avg': below_mean_cities,
    'readings_above_60': above_60_count
}

with open('aqi_insights.json', 'w', encoding='utf-8') as f:
    json.dump(insights, f, indent=2)

print('most volatile:', most_volatile_city, volatility[most_volatile_city])
print('below mean:', below_mean_cities)
print('above 60 count:', above_60_count)

most volatile: delhi 36.42
below mean: ['ahmedabad', 'surat', 'vadodara', 'rajkot', 'mumbai', 'pune', 'bengaluru', 'chennai']
above 60 count: 85


In [2]:
def get_band(val):
    if val <= 30:
        return 'Good'
    elif val <= 60:
        return 'Satisfactory'
    elif val <= 90:
        return 'Moderate'
    elif val <= 120:
        return 'Poor'
    else:
        return 'Very poor'

sorted_for_rank = sorted(tracker_rows, key=lambda r: float(r['avg_pm25']), reverse=True)

ranking_rows = []
for idx, r in enumerate(sorted_for_rank, start=1):
    avg_val = float(r['avg_pm25'])
    ranking_rows.append({
        'rank': idx,
        'city': r['city'],
        'avg_pm25': avg_val,
        'band': get_band(avg_val)
    })

with open('aqi_ranking.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['rank', 'city', 'avg_pm25', 'band'])
    writer.writeheader()
    writer.writerows(ranking_rows)

for top_r in ranking_rows[:3]:
    print(top_r)

{'rank': 1, 'city': 'delhi', 'avg_pm25': 170.18, 'band': 'Very poor'}
{'rank': 2, 'city': 'kolkata', 'avg_pm25': 101.13, 'band': 'Poor'}
{'rank': 3, 'city': 'ahmedabad', 'avg_pm25': 64.83, 'band': 'Moderate'}
